# Least-norm inversion: plain L² vs weighted L²(r²) [FAST]

Minimal demonstration using small basis (N=10) and coarse integration (n_points=500)
for fast iteration while developing.

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, 'utils')

from full_spectrum_utils import (
    RadialSpecs, build_block_forward,
    enumerate_blocks, block_data_split,
)
from normal_mode_kernel_utils import NormalModeDataRegistry, NormalModeKernelCatalog
from pygeoinf import LinearMinimumNormInversion
from pygeoinf.forward_problem import LinearForwardProblem
from pygeoinf.linear_solvers import CholeskySolver

print('Imports OK')

In [ ]:
# ── Load data and kernels (may take 1-2 min) ────────────────────────────────
import time
t0 = time.time()

DATA_DIR   = 'data/normal-mode-data'
KERNEL_DIR = 'data/normal-mode-kernels/kernels-all_PREM-layers_Adrian'

print('Loading registry...', end='', flush=True)
reg = NormalModeDataRegistry(DATA_DIR, dataset='arwen-paula')
print(f' done ({time.time()-t0:.1f}s)')

print('Loading catalog...', end='', flush=True)
t1 = time.time()
catalog = NormalModeKernelCatalog(KERNEL_DIR)
print(f' done ({time.time()-t1:.1f}s)')

S_MAX = 0  # Just first splitting degree for speed
blocks = enumerate_blocks(reg, S_MAX)
split  = block_data_split(reg, blocks)

block = blocks[0]
print(f'Using block s={block.s}, t={block.t}  ({len(split[block])} observations)')

In [ ]:
# ── Build forwards (FAST: N=10, n_points=500) ───────────────────────────────
# Default RadialSpecs uses n_points=2000 → too slow for iteration
# Reduce to N=10 basis, n_points=500 integration for 10× speedup

from intervalinf import LebesgueIntegrationConfig, IntegrationConfig

N = 10  # Small basis
cfg = IntegrationConfig(method='trapz', n_points=500)  # Coarse
lebesgue_cfg = LebesgueIntegrationConfig(inner_product=cfg, dual=cfg, general=cfg)

specs_flat = RadialSpecs(n_basis=N, weighted=False, lebesgue_cfg=lebesgue_cfg)
specs_w    = RadialSpecs(n_basis=N, weighted=True,  lebesgue_cfg=lebesgue_cfg)

print('Building forwards...')
t0 = time.time()

G_flat, C_D_flat, M_flat, D = build_block_forward(
    block.s, block.t, split[block], catalog, specs_flat)
print(f'  Flat:     {time.time()-t0:.1f}s')

t1 = time.time()
G_w, C_D_w, M_w, _ = build_block_forward(
    block.s, block.t, split[block], catalog, specs_w)
print(f'  Weighted: {time.time()-t1:.1f}s')

d_obs = split[block].data_vector
print(f'Data: {len(d_obs)} observations')

In [ ]:
# ── Least-norm solve ────────────────────────────────────────────────────────
print('Setting up LinearMinimumNormInversion...')
t0 = time.time()

solver = CholeskySolver()

fp_flat = LinearForwardProblem(G_flat, data_error_measure=C_D_flat)
fp_w    = LinearForwardProblem(G_w,    data_error_measure=C_D_w)

mn_flat = LinearMinimumNormInversion(fp_flat).minimum_norm_operator(solver)
print(f'Flat operator: {time.time()-t0:.1f}s')

t1 = time.time()
mn_w    = LinearMinimumNormInversion(fp_w   ).minimum_norm_operator(solver)
print(f'Weighted operator: {time.time()-t1:.1f}s')

print('\nComputing solutions...')
t0 = time.time()
m_flat = mn_flat(d_obs)
print(f'  Flat:     {time.time()-t0:.1f}s  ||m||={M_flat.norm(m_flat):.4e}')

t1 = time.time()
m_w    = mn_w(d_obs)
print(f'  Weighted: {time.time()-t1:.1f}s  ||m||_r²={M_w.norm(m_w):.4e}')

In [ ]:
# ── Quick profile: which mode is faster? ────────────────────────────────────
print('\n' + '='*60)
print('PROFILE: Flat vs Weighted (10 trials, averaged)')
print('='*60)

import timeit
flat_times = timeit.repeat(lambda: mn_flat(d_obs), number=1, repeat=3)
w_times    = timeit.repeat(lambda: mn_w(d_obs),    number=1, repeat=3)

print(f'Flat mode:     {np.mean(flat_times):.3f}±{np.std(flat_times):.3f} s')
print(f'Weighted mode: {np.mean(w_times):.3f}±{np.std(w_times):.3f} s')
print(f'Ratio (w/flat): {np.mean(w_times)/np.mean(flat_times):.2f}x')

In [ ]:
# ── Compare solutions ───────────────────────────────────────────────────────
R = specs_flat.earth_radius_km
r_test = np.array([1000., 2000., 3000., 4000., 5000.])

print(f"\n{'r (km)':>8}  {'vp_flat':>12}  {'vp_w':>12}  {'ratio':>10}")
for r0 in r_test:
    vp_f = m_flat[0][0](r0)
    vp_w = m_w[0][0](r0)
    ratio = vp_f / vp_w if abs(vp_w) > 1e-20 else float('inf')
    print(f"{r0:>8.0f}  {vp_f:>12.4e}  {vp_w:>12.4e}  {ratio:>10.2f}")

# Data fit
d_pred_flat = G_flat(m_flat)
d_pred_w    = G_w(m_w)
err = split[block].error_vector

chi_flat = float(np.sqrt(np.mean(((d_obs - d_pred_flat)/err)**2)))
chi_w    = float(np.sqrt(np.mean(((d_obs - d_pred_w   )/err)**2)))

print(f"\nRMS χ (flat):     {chi_flat:.3f}")
print(f"RMS χ (weighted): {chi_w:.3f}")
print(f"Target: χ ≈ 1 (discrepancy principle)")